# Clothing Classifier — Training

Transfer learning with `timm` (ResNet-18). Runs on CPU for local dev and moves
to GPU unchanged (the device is auto-detected). 15 clothing classes, 500 images each.

## 1. Imports

In [2]:
import os
from collections import Counter

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

import timm
from timm.data import resolve_data_config, create_transform

c:\Users\daru1\OneDrive\Desktop\Portfolio\clothing-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration
Every tunable knob lives here — nothing to hunt for in the cells below.

In [3]:
# --- data ---
DATA_DIR   = r"C:\Users\daru1\OneDrive\Desktop\Portfolio\Clothes_Dataset"
CKPT_DIR   = "checkpoints"

# --- model ---
MODEL_NAME = "convnext_tiny"      # bump to "resnet50" / "convnext_tiny" on GPU

# --- training ---
BATCH_SIZE      = 32
VAL_SPLIT       = 0.2
SEED            = 42
FREEZE_BACKBONE = True       # True = train only the head (fast on CPU)
EPOCHS          = 3          # 3 for the frozen CPU smoke-test; 15-30 on GPU
LR              = 1e-3 if FREEZE_BACKBONE else 1e-4
WEIGHT_DECAY    = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 3. Dataset & class inspection
Confirm the folder layout: one subfolder per class, balanced counts.

In [4]:
base = datasets.ImageFolder(DATA_DIR)
print("classes found:", len(base.classes))
print("total images :", len(base))
print()
counts = Counter(base.targets)
for i, cls in enumerate(base.classes):
    print(f"{cls:<25} {counts[i]}")

classes found: 15
total images : 7500

Blazer                    500
Celana_Panjang            500
Celana_Pendek             500
Gaun                      500
Hoodie                    500
Jaket                     500
Jaket_Denim               500
Jaket_Olahraga            500
Jeans                     500
Kaos                      500
Kemeja                    500
Mantel                    500
Polo                      500
Rok                       500
Sweter                    500


## 4. Model & transforms
Transforms are pulled from the model so normalization always matches the backbone.

In [5]:
num_classes = len(base.classes)
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)

cfg = resolve_data_config({}, model=model)
train_tf = create_transform(**cfg, is_training=True)    # with augmentation
val_tf   = create_transform(**cfg, is_training=False)   # clean, for eval

model = model.to(device)
print("input config:", cfg)

input config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.95, 'crop_mode': 'center'}


## 5. Train / validation split & DataLoaders
Stratified 80/20 split — keeps 100 of each class in validation.

In [6]:
train_idx, val_idx = train_test_split(
    range(len(base.targets)),
    test_size=VAL_SPLIT,
    stratify=base.targets,
    random_state=SEED,
)

train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("train:", len(train_ds), " val:", len(val_ds))   # expect 6000 / 1500

train: 6000  val: 1500


## 6. Sanity-check one batch
Catch any image-reading problem in seconds, before a full epoch.

In [7]:
imgs, labels = next(iter(train_loader))
print("batch images:", imgs.shape)   # expect [32, 3, 224, 224]
print("batch labels:", labels[:8])

batch images: torch.Size([32, 3, 224, 224])
batch labels: tensor([ 9, 12,  1,  6,  6, 10, 11,  6])


## 7. Training setup
Freeze policy, loss, optimizer.

In [24]:
# freeze everything, then re-enable just the classifier head
if FREEZE_BACKBONE:
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

print(f"trainable params: {sum(p.numel() for p in params):,}")

trainable params: 11,535


## 8. Training loop

In [8]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            seen += imgs.size(0)
    return total_loss / seen, correct / seen

In [26]:
best_acc = 0.0
os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"epoch {epoch}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, os.path.join(CKPT_DIR, "best.pt"))
        print(f"  saved best (val acc {best_acc:.3f})")

print("done. best val acc:", round(best_acc, 3))

epoch 1/3 | train loss 1.183 acc 0.615 | val loss 0.751 acc 0.746
  saved best (val acc 0.746)


epoch 2/3 | train loss 0.918 acc 0.698 | val loss 0.717 acc 0.756
  saved best (val acc 0.756)


epoch 3/3 | train loss 0.866 acc 0.705 | val loss 0.701 acc 0.757
  saved best (val acc 0.757)
done. best val acc: 0.757


## 9. Evaluation
_Next: confusion matrix + per-class report on `best.pt`._

In [27]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [28]:
UNFREEZE_EPOCHS = 5
FT_LR = 1e-4   

In [29]:
for p in model.parameters():
    p.requires_grad = True

In [30]:
# fresh optimizer over ALL params (the old one only knew about the head)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)

# cosine schedule: LR eases down toward 0 over the run for a cleaner finish
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCHS)

for epoch in range(1, UNFREEZE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"[finetune] epoch {epoch}/{UNFREEZE_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, "checkpoints/best.pt")
        print(f"  saved best (val acc {best_acc:.3f})")

print("finetune done. best val acc:", round(best_acc, 3))

  0%|          | 0/188 [00:00<?, ?it/s]

[finetune] epoch 1/5 | train loss 1.135 acc 0.627 | val loss 0.727 acc 0.757


[finetune] epoch 2/5 | train loss 0.827 acc 0.720 | val loss 0.661 acc 0.769
  saved best (val acc 0.769)


[finetune] epoch 3/5 | train loss 0.646 acc 0.780 | val loss 0.665 acc 0.783
  saved best (val acc 0.783)


[finetune] epoch 4/5 | train loss 0.490 acc 0.831 | val loss 0.595 acc 0.799
  saved best (val acc 0.799)


[finetune] epoch 5/5 | train loss 0.359 acc 0.876 | val loss 0.550 acc 0.821
  saved best (val acc 0.821)
finetune done. best val acc: 0.821


In [9]:
# load your 0.821 weights back into the fresh model
ckpt = torch.load("checkpoints/best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)

# unfreeze everything for full fine-tuning
for p in model.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss()   # run_epoch needs this
best_acc = 0.821                    # so phase 3 only overwrites best.pt on a real improvement
print("loaded checkpoint, all params unfrozen")

loaded checkpoint, all params unfrozen


In [11]:
# --- Phase 3: keep fine-tuning, longer schedule ---
MORE_EPOCHS = 3
FT_LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MORE_EPOCHS)

best_loss = float("inf")   # <-- initialize (or set to your best-so-far val loss, ~0.550)

for epoch in range(1, MORE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"[phase3] epoch {epoch}/{MORE_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc and va_loss < best_loss:   # <-- va_loss, not val_loss
        best_acc = va_acc
        best_loss = va_loss                          # <-- update BOTH
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, "checkpoints/best.pt")
        print(f"  saved best (val acc {best_acc:.3f}, loss {best_loss:.3f})")

print("phase3 done. best val acc:", round(best_acc, 3))

  0%|          | 0/188 [00:00<?, ?it/s]

[phase3] epoch 1/3 | train loss 0.678 acc 0.769 | val loss 0.653 acc 0.774


[phase3] epoch 2/3 | train loss 0.513 acc 0.823 | val loss 0.648 acc 0.791


[phase3] epoch 3/3 | train loss 0.336 acc 0.884 | val loss 0.625 acc 0.817
phase3 done. best val acc: 0.821


In [12]:
print(model)   # full layer tree

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True, bias=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), paddi

In [13]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=base.classes))
print(confusion_matrix(all_labels, all_preds))

                precision    recall  f1-score   support

        Blazer       0.81      0.73      0.77       100
Celana_Panjang       0.81      0.79      0.80       100
 Celana_Pendek       0.95      0.92      0.93       100
          Gaun       0.77      0.89      0.82       100
        Hoodie       0.88      0.93      0.90       100
         Jaket       0.67      0.67      0.67       100
   Jaket_Denim       0.86      0.92      0.89       100
Jaket_Olahraga       0.76      0.64      0.70       100
         Jeans       0.87      0.91      0.89       100
          Kaos       0.86      0.91      0.88       100
        Kemeja       0.78      0.69      0.73       100
        Mantel       0.84      0.79      0.81       100
          Polo       0.69      0.75      0.72       100
           Rok       0.86      0.81      0.84       100
        Sweter       0.85      0.90      0.87       100

      accuracy                           0.82      1500
     macro avg       0.82      0.82      0.82 